#Install Dependencies

In [ ]:
!pip install transformers sentencepiece torch -q

In [ ]:
#Import Libraries
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

In [ ]:
#Load Models (Optimized)
# English ↔ Hindi
en_hi_model_name = "Helsinki-NLP/opus-mt-en-hi"
hi_en_model_name = "Helsinki-NLP/opus-mt-hi-en"

# Multilingual (for Telugu)
multi_model_name = "facebook/nllb-200-distilled-600M"

# Load tokenizers & models
en_hi_tokenizer = AutoTokenizer.from_pretrained(en_hi_model_name)
en_hi_model = AutoModelForSeq2SeqLM.from_pretrained(en_hi_model_name)

hi_en_tokenizer = AutoTokenizer.from_pretrained(hi_en_model_name)
hi_en_model = AutoModelForSeq2SeqLM.from_pretrained(hi_en_model_name)

multi_tokenizer = AutoTokenizer.from_pretrained(multi_model_name)
multi_model = AutoModelForSeq2SeqLM.from_pretrained(multi_model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/812k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [ ]:
#Translation Function
def translate(text, direction):

    # ENG → HIN
    if direction == "en-hi":
        inputs = en_hi_tokenizer(text, return_tensors="pt", padding=True)
        outputs = en_hi_model.generate(**inputs)
        return en_hi_tokenizer.decode(outputs[0], skip_special_tokens=True)

    # HIN → ENG
    elif direction == "hi-en":
        inputs = hi_en_tokenizer(text, return_tensors="pt", padding=True)
        outputs = hi_en_model.generate(**inputs)
        return hi_en_tokenizer.decode(outputs[0], skip_special_tokens=True)

    # ENG → TEL (NLLB)
    elif direction == "en-te":
        multi_tokenizer.src_lang = "eng_Latn"
        inputs = multi_tokenizer(text, return_tensors="pt")
        outputs = multi_model.generate(**inputs, forced_bos_token_id=multi_tokenizer.convert_tokens_to_ids("tel_Telu"))
        return multi_tokenizer.decode(outputs[0], skip_special_tokens=True)

    # TEL → ENG
    elif direction == "te-en":
        multi_tokenizer.src_lang = "tel_Telu"
        inputs = multi_tokenizer(text, return_tensors="pt")
        outputs = multi_model.generate(**inputs, forced_bos_token_id=multi_tokenizer.convert_tokens_to_ids("eng_Latn"))
        return multi_tokenizer.decode(outputs[0], skip_special_tokens=True)

    else:
        return "Invalid direction"

In [ ]:
#Test the Model
print("ENG → HIN:", translate("Hello, how are you?", "en-hi"))
print("HIN → ENG:", translate("आप कैसे हैं?", "hi-en"))

print("ENG → TEL:", translate("India is a great country", "en-te"))
print("TEL → ENG:", translate("నువ్వు ఎలా ఉన్నావు", "te-en"))

ENG → HIN: हैलो, तुम कैसे हो?
HIN → ENG: How are you?
ENG → TEL: భారతదేశం గొప్ప దేశం
TEL → ENG: How are you?


In [ ]:
# Create folders and save models

en_hi_model.save_pretrained("models/en_hi/")
en_hi_tokenizer.save_pretrained("models/en_hi/")

hi_en_model.save_pretrained("models/hi_en/")
hi_en_tokenizer.save_pretrained("models/hi_en/")

multi_model.save_pretrained("models/multi/")
multi_tokenizer.save_pretrained("models/multi/")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('models/multi/tokenizer_config.json', 'models/multi/tokenizer.json')

In [ ]:
from google.colab import files

!zip -r translator_models.zip models

files.download("translator_models.zip")

updating: models/ (stored 0%)
updating: models/hi_en/ (stored 0%)
updating: models/hi_en/vocab.json (deflated 76%)
updating: models/hi_en/tokenizer_config.json (deflated 67%)
updating: models/hi_en/source.spm (deflated 60%)
updating: models/hi_en/generation_config.json (deflated 43%)
updating: models/hi_en/model.safetensors (deflated 7%)
updating: models/hi_en/config.json (deflated 63%)
updating: models/hi_en/target.spm (deflated 51%)
updating: models/en_hi/ (stored 0%)
updating: models/en_hi/vocab.json (deflated 76%)
updating: models/en_hi/tokenizer_config.json (deflated 67%)
updating: models/en_hi/source.spm (deflated 51%)
updating: models/en_hi/generation_config.json (deflated 43%)
updating: models/en_hi/model.safetensors (deflated 7%)
updating: models/en_hi/config.json (deflated 63%)
updating: models/en_hi/target.spm (deflated 60%)
updating: models/multi/ (stored 0%)
updating: models/multi/tokenizer_config.json (deflated 50%)
updating: models/multi/generation_config.json (deflated 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>